# Original vs Enhanced Detection — Adaptive Road Vision

Evaluate the **same held-out test images** with the selected YOLO11n or YOLO11s detector on original versus condition-enhanced images.

- Uses the final detector selection report produced by Notebook 21 (with compatible fallback filenames).
- Uses only test images and their YOLO labels.
- Day/Unknown images remain unchanged; Night/Rain/Fog use their corresponding trained enhancement checkpoint.
- Fails clearly when required checkpoints or labels are missing rather than silently reporting misleading metrics.

**Metric note:** this notebook reports fixed-threshold precision/recall/F1 at IoU 0.50. These are not Ultralytics mAP metrics. Enhancement checkpoints were trained using heuristic pseudo-targets, not verified clean ground truth.


In [ ]:
from pathlib import Path
import os, json, csv
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_dataset_root():
    for cand in [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]:
        if (cand / "metadata" / "final_dataset_metadata.csv").exists():
            return cand
        if (cand / "labels" / "train").exists() and any((cand / "labels" / "train").glob("*.txt")):
            return cand
    return PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"

FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
METADATA_CANDIDATES = [
    DATASET_ROOT / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2" / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
METADATA_PATH = next((p for p in METADATA_CANDIDATES if p.exists()), METADATA_CANDIDATES[0])
SELECTION_CANDIDATES = [
    PROJECT_ROOT / "results" / "detection" / "final_detector_selection.json",
    PROJECT_ROOT / "results" / "detection" / "final_detector_selection_report.json",
    PROJECT_ROOT / "results" / "detection" / "detector_selection.json",
]
SELECTED_PATH = next((p for p in SELECTION_CANDIDATES if p.exists()), SELECTION_CANDIDATES[0])
MODEL_ROOT = PROJECT_ROOT / "models"
RESULT_ROOT = PROJECT_ROOT / "results" / "detection"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Device:", TORCH_DEVICE)


## 1. Load the selected detector and test metadata

In [ ]:
if not SELECTED_PATH.exists():
    raise FileNotFoundError(
        f"Final detector selection report not found. Run Notebook 21 first. Checked: "
        + ", ".join(str(p) for p in SELECTION_CANDIDATES)
    )

selection = json.loads(SELECTED_PATH.read_text(encoding="utf-8"))
selected_detector = selection.get("selected_detector", "Unknown")
selected_model = str(selection.get("selected_model", "")).upper()
checkpoint_value = (selection.get("stable_downstream_checkpoint")
                    or selection.get("selected_checkpoint")
                    or selection.get("checkpoint"))
if not checkpoint_value:
    raise KeyError("Selection report has no stable_downstream_checkpoint / selected_checkpoint / checkpoint field.")
selected_checkpoint = Path(checkpoint_value)
if not selected_checkpoint.is_absolute():
    selected_checkpoint = PROJECT_ROOT / selected_checkpoint
if not selected_checkpoint.exists():
    raise FileNotFoundError(f"Selected detector checkpoint does not exist: {selected_checkpoint}. Re-run Notebook 21 after training.")
if selected_model and selected_model not in {"YOLO11N", "YOLO11S", "YOLO11N.PT", "YOLO11S.PT"}:
    raise ValueError(f"Expected selected YOLO11n or YOLO11s, found: {selected_model}")

print("Selection report:", SELECTED_PATH)
print("Selected detector:", selected_detector)
print("Model:", selected_model)
print("Checkpoint:", selected_checkpoint)

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset metadata: {METADATA_PATH}")
metadata = pd.read_csv(METADATA_PATH)
if "split" not in metadata.columns:
    raise KeyError("Metadata must contain a 'split' column.")
if "project_condition" not in metadata.columns:
    if "condition" in metadata.columns:
        metadata["project_condition"] = metadata["condition"]
    elif "weather" in metadata.columns:
        metadata["project_condition"] = metadata["weather"]
    else:
        metadata["project_condition"] = "Unknown"
if "image_name" not in metadata.columns and "image_path" in metadata.columns:
    metadata["image_name"] = metadata["image_path"].map(lambda x: Path(str(x)).name)
if "image_name" not in metadata.columns:
    raise KeyError("Metadata must contain image_name or image_path.")
test_meta = metadata[metadata["split"].astype(str).str.strip().str.lower().isin(["test", "testing"])].copy()
if test_meta.empty:
    raise ValueError("No test rows found in metadata; expected split='test'.")
test_meta["project_condition"] = test_meta["project_condition"].fillna("Unknown").astype(str).str.strip().str.title()
test_meta.loc[~test_meta["project_condition"].isin(["Day", "Night", "Rain", "Fog", "Unknown"]), "project_condition"] = "Unknown"
print("Test metadata rows:", len(test_meta))
print(test_meta["project_condition"].value_counts(dropna=False))


## 2. Load the detector

Ultralytics is used for inference and evaluation. The detector checkpoint is the selected checkpoint created by Notebook 21.


In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError("Install Ultralytics first: pip install ultralytics")

detector = YOLO(str(selected_checkpoint))
print("Loaded:", selected_checkpoint)


## 3. Load enhancement models

This cell reconstructs the compact residual encoder-decoder used in notebooks 14–16. The model interface is compatible with the saved enhancement checkpoints.


In [ ]:
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels, affine=True),
        )

    def forward(self, x):
        return x + self.block(x)


class ConditionEnhancer(nn.Module):
    """
    Compact residual encoder-decoder matching the architecture
    trained in Notebooks 14 (Night), 15 (Rain), and 16 (Fog).
    """
    def __init__(self, base_channels=32, num_res_blocks=4):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels * 2, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels * 4, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.res = nn.Sequential(*[ResidualBlock(base_channels * 4) for _ in range(num_res_blocks)])
        self.dec2 = nn.Sequential(
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.out = nn.Conv2d(base_channels, 3, 3, padding=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        z = self.res(e3)

        z = F.interpolate(z, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        z = self.dec2(z) + e2

        z = F.interpolate(z, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        z = self.dec1(z) + e1

        residual = torch.tanh(self.out(z)) * 0.5
        return torch.clamp(x + residual, 0.0, 1.0)


# Backward compatibility alias
EnhancementNet = ConditionEnhancer


def load_checkpoint_model(path):
    ckpt = torch.load(path, map_location=TORCH_DEVICE)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
    elif isinstance(ckpt, dict):
        state = ckpt
    else:
        state = ckpt.state_dict()

    cleaned = {}
    for k, v in state.items():
        cleaned[k.replace("module.", "")] = v

    model = ConditionEnhancer().to(TORCH_DEVICE)
    model.load_state_dict(cleaned, strict=True)
    model.eval()
    return model


enhancer_paths = {
    "Night": MODEL_ROOT / "enhancement" / "night" / "night_enhancer_best.pth",
    "Rain": MODEL_ROOT / "enhancement" / "rain" / "rain_enhancer_best.pth",
    "Fog": MODEL_ROOT / "enhancement" / "fog" / "fog_enhancer_best.pth",
}

enhancers = {}

for condition, path in enhancer_paths.items():
    if path.exists():
        try:
            enhancers[condition] = load_checkpoint_model(path)
            print(f"Loaded {condition}: {path}")
        except Exception as e:
            print(f"Could not load {condition} enhancer: {e}")
    else:
        print(f"Missing {condition} enhancer: {path}")

print("Loaded enhancement models:", list(enhancers.keys()))
required_enhancers = set(test_meta["project_condition"].unique()) & {"Night", "Rain", "Fog"}
missing_enhancers = sorted(required_enhancers - set(enhancers))
if missing_enhancers:
    missing_paths = [str(enhancer_paths[c]) for c in missing_enhancers]
    raise FileNotFoundError(
        "Required enhancement checkpoint(s) missing or incompatible for test conditions "
        f"{missing_enhancers}. Expected files: {missing_paths}. Train the relevant Notebook 14/15/16 first."
    )


## 4. Build the test-image evaluation set

Only test images are used. The original image is always retained for the baseline inference.


In [ ]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def resolve_test_image(row):
    candidates = []

    for key in ["image_path", "image_name"]:
        if key in row.index and pd.notna(row[key]):
            value = str(row[key])
            p = Path(value)
            if p.is_absolute():
                candidates.append(p)
            else:
                candidates.extend([
                    PROJECT_ROOT / value,
                    DATASET_ROOT / value,
                    DATASET_ROOT / "images" / "test" / value,
                    DATASET_ROOT / "images" / "testing" / value,
                ])

    image_name = str(row.get("image_name", ""))
    if image_name:
        candidates.append(DATASET_ROOT / "images" / "test" / image_name)
        candidates.append(DATASET_ROOT / "images" / "testing" / image_name)

    stem = Path(image_name).stem
    if stem:
        for ext in IMAGE_EXTENSIONS:
            candidates.append(DATASET_ROOT / "images" / "test" / (stem + ext))
            candidates.append(DATASET_ROOT / "images" / "testing" / (stem + ext))

    for p in candidates:
        if p.exists():
            return p

    return None


test_meta["resolved_image_path"] = test_meta.apply(resolve_test_image, axis=1)
missing_paths = test_meta["resolved_image_path"].isna().sum()

print("Resolvable test images:", len(test_meta) - missing_paths)
print("Missing image paths:", missing_paths)

if missing_paths:
    display(test_meta[test_meta["resolved_image_path"].isna()].head(10))
    raise FileNotFoundError(f"{missing_paths} test images could not be resolved. Fix metadata paths before evaluation.")


## 5. Enhancement preprocessing

Images are resized to the enhancer input size and converted back to full-resolution-like RGB tensors before detection. The detector receives the enhanced image array; the original image is used for the baseline.


In [ ]:
ENHANCE_SIZE = 256

to_tensor = transforms.Compose([
    transforms.Resize((ENHANCE_SIZE, ENHANCE_SIZE)),
    transforms.ToTensor(),
])

def enhance_pil(image, condition):
    if condition not in enhancers:
        return image.copy()

    tensor = to_tensor(image).unsqueeze(0).to(TORCH_DEVICE)

    with torch.inference_mode():
        enhanced = enhancers[condition](tensor)[0].cpu()

    enhanced = transforms.ToPILImage()(enhanced)
    enhanced = enhanced.resize(image.size, Image.Resampling.BILINEAR)
    return enhanced


def condition_for_row(row):
    value = str(row.get("project_condition", "Unknown"))
    return value if value in {"Day", "Night", "Rain", "Fog", "Unknown"} else "Unknown"


## 6. Detection helper

Detection metrics are calculated from the ground-truth YOLO labels. Predictions are matched to ground truth using IoU ≥ 0.50 for class-aware one-to-one matching.

This provides an apples-to-apples baseline between original and enhanced images using the same detector and test labels.


In [ ]:
def load_yolo_labels(label_path, image_width, image_height):
    boxes = []
    classes = []

    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.int64)

    for line in text.splitlines():
        p = line.split()
        if len(p) != 5:
            continue

        try:
            cls_float = float(p[0])
            coords = np.asarray(list(map(float, p[1:])), dtype=np.float64)
        except (ValueError, OverflowError):
            continue
        if not np.isfinite(cls_float) or not np.all(np.isfinite(coords)) or not cls_float.is_integer():
            continue
        cls = int(cls_float)
        cx, cy, w, h = coords.tolist()
        if cls < 0 or w <= 0 or h <= 0:
            continue
        # Reject malformed normalized YOLO boxes instead of letting them distort metrics.
        if np.any(coords < 0) or np.any(coords > 1):
            continue
        names = getattr(detector, "names", None)
        if names is not None and cls not in set(int(k) for k in names.keys()):
            continue

        # YOLO labels are normalized [0, 1]; detector predictions are pixel xyxy.
        x1 = (cx - w / 2) * image_width
        y1 = (cy - h / 2) * image_height
        x2 = (cx + w / 2) * image_width
        y2 = (cy + h / 2) * image_height

        boxes.append([x1, y1, x2, y2])
        classes.append(cls)

    return np.asarray(boxes, dtype=np.float32), np.asarray(classes, dtype=np.int64)


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)

    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]

    ix1 = np.maximum(ax1, bx1)
    iy1 = np.maximum(ay1, by1)
    ix2 = np.minimum(ax2, bx2)
    iy2 = np.minimum(ay2, by2)

    iw = np.maximum(0, ix2 - ix1)
    ih = np.maximum(0, iy2 - iy1)
    inter = iw * ih

    area_a = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    area_b = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)

    return inter / np.maximum(area_a + area_b - inter, 1e-9)


def match_predictions(gt_boxes, gt_cls, pred_boxes, pred_cls, pred_conf, iou_thr=0.50):
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes), []

    order = np.argsort(-pred_conf)
    pred_boxes = pred_boxes[order]
    pred_cls = pred_cls[order]
    pred_conf = pred_conf[order]

    ious = iou_matrix(pred_boxes, gt_boxes)
    used_gt = set()
    matches = []
    tp = 0

    for pi in range(len(pred_boxes)):
        candidates = np.where(gt_cls == pred_cls[pi])[0]
        candidates = [g for g in candidates if g not in used_gt]

        if not candidates:
            continue

        best_g = max(candidates, key=lambda g: ious[pi, g])

        if ious[pi, best_g] >= iou_thr:
            used_gt.add(best_g)
            tp += 1
            matches.append({
                "pred_index": int(order[pi]),
                "gt_index": int(best_g),
                "iou": float(ious[pi, best_g]),
                "class_id": int(pred_cls[pi]),
                "confidence": float(pred_conf[pi]),
            })

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp

    return tp, fp, fn, matches


## 7. Run original vs enhanced inference

For speed, this cell evaluates the test set in batches through repeated Ultralytics inference calls. Confidence threshold is fixed for both conditions.


In [ ]:
CONF_THRESHOLD = 0.25
IOU_MATCH_THRESHOLD = 0.50
IMG_SIZE_DETECTOR = 640

def predict_image(pil_image):
    results = detector.predict(
        source=np.asarray(pil_image),
        imgsz=IMG_SIZE_DETECTOR,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )

    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        return (
            np.zeros((0, 4), dtype=np.float32),
            np.zeros((0,), dtype=np.int64),
            np.zeros((0,), dtype=np.float32),
        )

    boxes = result.boxes.xyxy.cpu().numpy().astype(np.float32)
    classes = result.boxes.cls.cpu().numpy().astype(np.int64)
    conf = result.boxes.conf.cpu().numpy().astype(np.float32)

    return boxes, classes, conf


rows = []
max_images = None  # Set an integer for a pilot run; None evaluates the full test set.

eval_meta = test_meta[test_meta["resolved_image_path"].notna()].copy()
if max_images is not None:
    eval_meta = eval_meta.head(max_images)

for idx, row in enumerate(eval_meta.itertuples(index=False), start=1):
    image_path = Path(row.resolved_image_path)
    condition = str(row.project_condition)

    image = Image.open(image_path).convert("RGB")
    enhanced_image = enhance_pil(image, condition)

    # Keep label lookup explicit and fail rather than treating missing labels as empty ground truth.
    label_path = DATASET_ROOT / "labels" / "test" / f"{image_path.stem}.txt"
    if not label_path.exists():
        label_path = DATASET_ROOT / "labels" / "testing" / f"{image_path.stem}.txt"
    if not label_path.exists():
        raise FileNotFoundError(
            f"Missing test label for {image_path.name}. Checked: "
            f"{DATASET_ROOT / 'labels' / 'test' / (image_path.stem + '.txt')} and "
            f"{DATASET_ROOT / 'labels' / 'testing' / (image_path.stem + '.txt')}. "
            "Fix dataset labels before evaluating; missing labels must not be counted as empty scenes."
        )
    gt_boxes, gt_cls = load_yolo_labels(label_path, image.width, image.height)

    for mode, current_image in [
        ("original", image),
        ("enhanced", enhanced_image),
    ]:
        pred_boxes, pred_cls, pred_conf = predict_image(current_image)

        tp, fp, fn, matches = match_predictions(
            gt_boxes, gt_cls,
            pred_boxes, pred_cls, pred_conf,
            iou_thr=IOU_MATCH_THRESHOLD
        )

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)

        rows.append({
            "image_id": getattr(row, "image_id", image_path.stem),
            "image_name": getattr(row, "image_name", image_path.name),
            "condition": condition,
            "mode": mode,
            "image_path": str(image_path),
            "gt_count": int(len(gt_boxes)),
            "pred_count": int(len(pred_boxes)),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        })

    if idx % 250 == 0:
        print(f"Processed {idx:,}/{len(eval_meta):,}")

per_image_df = pd.DataFrame(rows)

if per_image_df.empty:
    raise RuntimeError("No evaluation rows were produced.")

print("Evaluation rows:", len(per_image_df))
display(per_image_df.head())


## 8. Aggregate original vs enhanced metrics

In [ ]:
def aggregate_metrics(df):
    tp = df["tp"].sum()
    fp = df["fp"].sum()
    fn = df["fn"].sum()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)

    return pd.Series({
        "images": df["image_name"].nunique(),
        "gt_objects": int(df["gt_count"].sum()),
        "pred_objects": int(df["pred_count"].sum()),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })


overall_df = pd.DataFrame([
    {"mode": mode, **aggregate_metrics(group).to_dict()}
    for mode, group in per_image_df.groupby("mode", sort=False)
])

display(overall_df)

condition_df = pd.DataFrame([
    {"condition": condition, "mode": mode, **aggregate_metrics(group).to_dict()}
    for (condition, mode), group in per_image_df.groupby(["condition", "mode"], sort=False)
])

display(condition_df)


## 9. Original vs enhanced delta

In [ ]:
pivot = overall_df.set_index("mode")

if {"original", "enhanced"}.issubset(pivot.index):
    delta = pd.DataFrame({
        "original": pivot.loc["original"],
        "enhanced": pivot.loc["enhanced"],
        "delta_enhanced_minus_original": (
            pivot.loc["enhanced"] - pivot.loc["original"]
        ),
    })
    display(delta)
else:
    delta = pd.DataFrame()
    print("Both original and enhanced modes are required for delta analysis.")


In [ ]:
# Condition-specific F1 delta.
if not condition_df.empty:
    f1_pivot = condition_df.pivot_table(
        index="condition",
        columns="mode",
        values="f1",
        aggfunc="first"
    )

    if {"original", "enhanced"}.issubset(f1_pivot.columns):
        f1_pivot["delta_enhanced_minus_original"] = (
            f1_pivot["enhanced"] - f1_pivot["original"]
        )

    display(f1_pivot)


## 10. Visualize detection impact

In [ ]:
plot_df = condition_df.pivot_table(
    index="condition",
    columns="mode",
    values="f1",
    aggfunc="first"
)

if not plot_df.empty:
    plt.figure(figsize=(10, 5))
    plot_df.plot(kind="bar", ax=plt.gca())
    plt.ylabel("F1")
    plt.title("Original vs Enhanced Detection by Condition")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


In [ ]:
# Readable percentage view and explicit decision aid (does not claim statistical significance).
summary_view = overall_df.copy()
for metric in ["precision", "recall", "f1"]:
    summary_view[metric] = (summary_view[metric] * 100).round(2)
print("Overall metrics (% except counts):")
display(summary_view)

if {"original", "enhanced"}.issubset(set(overall_df["mode"])):
    metric_rows = overall_df.set_index("mode")
    f1_delta = float(metric_rows.loc["enhanced", "f1"] - metric_rows.loc["original", "f1"])
    print(f"Enhanced minus original F1: {f1_delta:+.4f} ({f1_delta*100:+.2f} percentage points)")
    if f1_delta > 1e-6:
        print("Interpretation: enhanced images improved this fixed-threshold F1 on this test set. Check each condition before concluding.")
    elif f1_delta < -1e-6:
        print("Interpretation: enhanced images reduced this fixed-threshold F1 on this test set. Check which conditions/classes caused the drop.")
    else:
        print("Interpretation: no measurable overall F1 change at the chosen confidence and IoU thresholds.")
print("Reminder: this custom fixed-threshold F1 is not mAP and does not establish statistical significance.")


## 11. Save evaluation artifacts

In [ ]:
per_image_path = RESULT_ROOT / "original_vs_enhanced_detection_per_image.csv"
overall_path = RESULT_ROOT / "original_vs_enhanced_detection_overall.csv"
condition_path = RESULT_ROOT / "original_vs_enhanced_detection_by_condition.csv"
report_path = RESULT_ROOT / "original_vs_enhanced_detection_report.json"

per_image_df.to_csv(per_image_path, index=False)
overall_df.to_csv(overall_path, index=False)
condition_df.to_csv(condition_path, index=False)

report = {
    "selected_detector": selected_detector,
    "selected_model": selected_model,
    "selected_checkpoint": str(selected_checkpoint),
    "confidence_threshold": CONF_THRESHOLD,
    "iou_match_threshold": IOU_MATCH_THRESHOLD,
    "detector_img_size": IMG_SIZE_DETECTOR,
    "enhancer_input_size": ENHANCE_SIZE,
    "test_images_evaluated": int(eval_meta["image_name"].nunique()),
    "test_images_missing_paths": int(missing_paths),
    "metric_definition": "Fixed confidence threshold precision/recall/F1 at IoU 0.50; not mAP.",
    "enhancement_models_loaded": sorted(enhancers.keys()),
    "overall_metrics": overall_df.to_dict(orient="records"),
    "condition_metrics": condition_df.to_dict(orient="records"),
    "note": (
        "Enhanced detection is evaluated as downstream system behavior. "
        "Enhancement training used pseudo-reference targets, so these "
        "results should not be interpreted as photographic ground truth."
    ),
}

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", per_image_path)
print("Saved:", overall_path)
print("Saved:", condition_path)
print("Saved:", report_path)


## Final interpretation

Use the saved metrics to determine whether condition-specific enhancement changes downstream detections. A positive F1 delta means a higher fixed-threshold F1 under this evaluation setup; a negative delta means lower F1. This custom metric is **not equivalent to YOLO mAP**. For a publication-quality comparison, also run Ultralytics validation on the same original and enhanced image sets with a consistent label mapping and report mAP@0.50 and mAP@0.50:0.95.

Check per-condition results: enhancement may help Night/Fog but hurt Rain or Day. Do not assume enhanced images are better merely because they look brighter.
